# Load Data and see schema

In [ ]:
!pip install -q kagglehub pandas polars duckdb matplotlib numpy

In [ ]:

import requests
import os
import pandas as pd
import polars as pl
import duckdb
import time
import matplotlib.pyplot as plt
import numpy as np

import kagglehub
from kagglehub import KaggleDatasetAdapter

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

from xgboost import XGBClassifier
# Source of data set https://www.kaggle.com/datasets/aryan208/financial-transactions-dataset-for-fraud-detection
path_df = kagglehub.dataset_download(
    "aryan208/financial-transactions-dataset-for-fraud-detection"
)

print("Dataset path:", path_df)
print("Files:", os.listdir(path_df))

# Load CSV
df_path = os.path.join(path_df, "financial_fraud_detection_dataset.csv")
df = pd.read_csv(df_path)

print("Dataset 4 shape:", df.shape)
df.head()

Using Colab cache for faster access to the 'financial-transactions-dataset-for-fraud-detection' dataset.
Dataset path: /kaggle/input/financial-transactions-dataset-for-fraud-detection
Files: ['financial_fraud_detection_dataset.csv']
Dataset 4 shape: (5000000, 18)


,transaction_id,timestamp,sender_account,receiver_account,amount,transaction_type,merchant_category,location,device_used,is_fraud,fraud_type,time_since_last_transaction,spending_deviation_score,velocity_score,geo_anomaly_score,payment_channel,ip_address,device_hash
0,T100000,2023-08-22T09:22:43.516168,ACC877572,ACC388389,343.78,withdrawal,utilities,Tokyo,mobile,False,NaN,NaN,-0.21,3,0.22,card,13.101.214.112,D8536477
1,T100001,2023-08-04T01:58:02.606711,ACC895667,ACC944962,419.65,withdrawal,online,Toronto,atm,False,NaN,NaN,-0.14,7,0.96,ACH,172.52.47.194,D2622631
2,T100002,2023-05-12T11:39:33.742963,ACC733052,ACC377370,2773.86,deposit,other,London,pos,False,NaN,NaN,-1.78,20,0.89,card,185.98.35.23,D4823498
3,T100003,2023-10-10T06:04:43.195112,ACC996865,ACC344098,1666.22,deposit,online,Sydney,pos,False,NaN,NaN,-0.60,6,0.37,wire_transfer,107.136.36.87,D9961380
4,T100004,2023-09-24T08:09:02.700162,ACC584714,ACC497887,24.43,transfer,utilities,Toronto,mobile,False,NaN,NaN,0.79,13,0.27,ACH,108.161.108.255,D7637601


In [ ]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

paysim = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "ealaxi/paysim1",
    "PS_20174392719_1491204439457_log.csv"
)

print("PaySim shape:", paysim.shape)
paysim.sample(5)

/tmp/ipykernel_24112/3232599887.py:4: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  paysim = kagglehub.load_dataset(


Using Colab cache for faster access to the 'paysim1' dataset.
PaySim shape: (6362620, 11)


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [ ]:
print(df['is_fraud'].value_counts())
print(df['is_fraud'].value_counts(normalize=True))
print(paysim['isFraud'].value_counts())
print(paysim['isFraud'].value_counts(normalize=True))

is_fraud
False    4820447
True      179553
Name: count, dtype: int64
is_fraud
False    0.964089
True     0.035911
Name: proportion, dtype: float64
isFraud
0    6354407
1       8213
Name: count, dtype: int64
isFraud
0    0.998709
1    0.001291
Name: proportion, dtype: float64


# DATASET CLEANING

In [ ]:
#Training dataset
# Remove duplicates
df = df.drop_duplicates()

# Fill missing values
df.fillna({
    'device_used': 'unknown',
    'location': 'unknown',
    'merchant_category': 'unknown'
}, inplace=True)

# Convert timestamp
df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')

# Ensure label is int
df['is_fraud'] = df['is_fraud'].astype(int)


df.head()

In [ ]:
# Testing dataset
# Remove duplicates
paysim = paysim.drop_duplicates()

# Convert step → timestamp
paysim['timestamp'] = pd.to_datetime("2025-01-01") + pd.to_timedelta(paysim['step'], unit='h')

# Rename columns
paysim.rename(columns={
    'type': 'transaction_type',
    'nameOrig': 'sender_account',
    'nameDest': 'receiver_account',
    'isFraud': 'is_fraud'
}, inplace=True)


In [ ]:
# Balance features
paysim['balance_change_sender'] = paysim['newbalanceOrig'] - paysim['oldbalanceOrg']
paysim['balance_change_receiver'] = paysim['newbalanceDest'] - paysim['oldbalanceDest']

# Fraud indicator
paysim['is_large_transaction'] = (paysim['amount'] > 200000).astype(int)

# Time features
paysim = paysim.sort_values(['sender_account', 'timestamp'])
paysim['time_since_last_tx'] = paysim.groupby('sender_account')['timestamp'].diff().dt.total_seconds()

In [ ]:
df = df.sort_values(['sender_account', 'timestamp'])

# Time feature
df['time_since_last_tx'] = df.groupby('sender_account')['timestamp'].diff().dt.total_seconds()

# Behavioral features
df['avg_amount_user'] = df.groupby('sender_account')['amount'].transform('mean')
df['amount_deviation'] = df['amount'] - df['avg_amount_user']

# Time pattern
df['hour'] = df['timestamp'].dt.hour
df['is_night'] = ((df['hour'] < 6) | (df['hour'] > 22)).astype(int)

In [ ]:
df.sample(10)


In [ ]:
# Add missing columns to PaySim
for col in ['merchant_category', 'location', 'device_used']:
    paysim[col] = 'unknown'

# Select common columns
common_cols = [
    'timestamp',
    'transaction_type',
    'amount',
    'sender_account',
    'receiver_account',
    'is_fraud',
    'time_since_last_tx'
]

df_sub = df[common_cols].copy()
paysim_sub = paysim[common_cols].copy()

# Add source column
df_sub['source'] = 'dataset'
paysim_sub['source'] = 'paysim'

# Merge
final_df = pd.concat([df_sub, paysim_sub], ignore_index=True)

# Final clean
final_df = final_df.fillna(0)

print("Final dataset shape:", final_df.shape)
final_df.head()

possible other data sets
https://huggingface.co/datasets/CiferAI/Cifer-Fraud-Detection-Dataset-AF has a flagged feature the see (can be better to use to see false postives)

https://www.kaggle.com/datasets/ealaxi/paysim1

https://github.com/necst/amaretto_dataset

## Comments from teacher
Look for how it was flagged\
if data was collected by human we can compare vs  our model\
check the time difference between the different entry\
make a bin predictor for fraud\
look in to  (lightgbm xgboost SHAP)


In [ ]:
print(df['timestaamp'].dtype)


# Model

Features

In [ ]:
full_features = [
    'amount',
    'time_since_last_tx',
    'spending_deviation_score',
    'velocity_score',
    'geo_anomaly_score'
]
common_features = [
    'amount',
    'time_since_last_tx'
]

In [ ]:
X_full = df[full_features]
y_full = df['is_fraud']

X_common = df[common_features]
y_common = df['is_fraud']

In [ ]:
X_train_full, X_test_full, y_train_full, y_test_full = train_test_split(
    X_full, y_full, test_size=0.2, random_state=42
)
X_train_common, X_test_common, y_train_common, y_test_common = train_test_split(
    X_common, y_common, test_size=0.2, random_state=42
)

Training

In [ ]:
model_A = XGBClassifier()

model_A.fit(X_train_full, y_train_full)

In [ ]:
model_B = XGBClassifier()

model_B.fit(X_train_common, y_train_common)

In [ ]:
y_pred_A = model_A.predict(X_test_full)

In [ ]:
y_pred_B = model_B.predict(X_test_common)

In [ ]:
print("MODEL A (FULL FEATURES)")
print("Accuracy:", accuracy_score(y_test_full, y_pred_A))
print("F1 Score:", f1_score(y_test_full, y_pred_A))

print("Confusion Matrix:")
print(confusion_matrix(y_test_full, y_pred_A))

print("Classification Report:")
print(classification_report(y_test_full, y_pred_A))

In [ ]:
print("\nMODEL B (COMMON FEATURES)")
print("Accuracy:", accuracy_score(y_test_common, y_pred_B))
print("F1 Score:", f1_score(y_test_common, y_pred_B))

print("Confusion Matrix:")
print(confusion_matrix(y_test_common, y_pred_B))

print("Classification Report:")
print(classification_report(y_test_common, y_pred_B))

In [ ]:

# Get feature importance from Model A
importance = model_A.feature_importances_

# Match with feature names
feature_importance_df = pd.DataFrame({
    'Feature': X_train_full.columns,
    'Importance': importance
})

# Sort
feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False)

print(feature_importance_df.head(10))

In [ ]:
feature_importance_df.head(10).plot(
    x='Feature',
    y='Importance',
    kind='barh'
)
plt.gca().invert_yaxis()
plt.show()